# Phase 2: stage the invest PersonMailingCountry backfill

Freezes the invest mailing-address populations into `crm_imp_person_accounts`,
one row per account, `_operation='update'`, batch `2026-08-26_invest_mailing_backfill`.

**This notebook only writes to the local MySQL staging table. It never touches Salesforce.**

Goal: `PersonMailingCountry` holds the ISO English country **name** ("Austria", not "AT").
Core rule: address data only moves as a **complete block from one source** — that is what
makes a Vienna-street/Germany-country mismatch impossible by construction.

| pop | definition | action |
|---|---|---|
| A | mailing block fully empty, `BillingStreet` set | copy full billing block, country as mapped name — **except** the ~108 rows whose billing code contradicts the postal pattern (wrong code in the source, see 05 query 3): those copy the address but suppress the country + land on a review CSV |
| B | block empty, no billing street, `BillingCountryCode__c` set | country name only |
| C | `PersonMailingCountry` is a bare ISO-2 code | in-place code→name conversion, address untouched |
| D | partial mailing block (street/city set, country empty) | **not staged** — review export |

Mapping: `country_names.COUNTRY_NAMES` (generated from datasets/country-codes + overrides).
Coverage is asserted — no silent fallback, unmapped codes abort the staging.

Prerequisites: fresh mirror (`python sf_objects_download.py` at repo root), then
`01_create_mirror_indexes.sql` re-applied. Recon: `05_recon_mailing_country.sql`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repo root: config, mysql_client
sys.path.insert(0, str(Path.cwd()))          # this dir: country_names

from config import load_mysql_config
from mysql_client import MySQLClient
from country_names import COUNTRY_NAMES

BATCH_ID = "2026-08-26_invest_mailing_backfill"

db = MySQLClient(load_mysql_config())
print("connected | batch:", BATCH_ID, "| mapping entries:", len(COUNTRY_NAMES))

## 1. Mirror freshness

`MAX(LastModifiedDate)` must be from today's refresh; if stale, stop and re-run
`sf_objects_download.py` before staging.

In [ ]:
row = db.fetch_one(
    "SELECT COUNT(*) AS n, MAX(LastModifiedDate) AS newest FROM crm_person_account_sfid_prod"
)
print(f"crm_person_account_sfid_prod  {row['n']:>12,}  newest: {row['newest']}")

## 2. Mapping table + coverage assert

The ISO-2 → name map is materialized as `crm_tmp_country_names` so the INSERTs
stay pure SQL. Then: every code the populations can produce must be a key —
unmapped codes (e.g. the non-ISO `FL`/`XX` seen population-wide) abort here
rather than being guessed.

In [ ]:
db.execute("DROP TABLE IF EXISTS crm_tmp_country_names")
db.execute("""
    CREATE TABLE crm_tmp_country_names (
        code VARCHAR(8)  NOT NULL PRIMARY KEY,
        name VARCHAR(80) NOT NULL
    )
""")
with db.transaction() as conn:
    for c, n in COUNTRY_NAMES.items():
        db.execute("INSERT INTO crm_tmp_country_names (code, name) VALUES (%s, %s)", (c, n), conn=conn)
print(f"crm_tmp_country_names: {db.fetch_one('SELECT COUNT(*) AS n FROM crm_tmp_country_names')['n']} codes")

# Every code in scope must map (billing codes feed A/B, mailing codes feed C).
unmapped = db.fetch_all("""
    SELECT code, SUM(n) AS n FROM (
        SELECT UPPER(BillingCountryCode__c) AS code, COUNT(*) AS n
        FROM crm_person_account_sfid_prod
        WHERE InvestCustomer__pc = 'True'
          AND BillingCountryCode__c IS NOT NULL AND BillingCountryCode__c <> ''
        GROUP BY 1
        UNION ALL
        SELECT UPPER(PersonMailingCountry), COUNT(*)
        FROM crm_person_account_sfid_prod
        WHERE InvestCustomer__pc = 'True'
          AND PersonMailingCountry REGEXP '^[A-Za-z]{2}$'
        GROUP BY 1
    ) x
    LEFT JOIN crm_tmp_country_names cn USING (code)
    WHERE cn.code IS NULL
    GROUP BY 1
""")
assert not unmapped, f"unmapped codes in scope: {unmapped}"
print("every code in scope is mapped")

## 3. Population counts (contract)

The recon numbers from `05_recon_mailing_country.sql` query 1 are the contract.
Small drift after a refresh is possible; investigate anything larger than a handful.

In [ ]:
# WHERE fragments, verbatim from 05_recon_mailing_country.sql query 1.
INVEST = "acc.InvestCustomer__pc = 'True'"
BLOCK_EMPTY = """
      (acc.PersonMailingStreet     IS NULL OR acc.PersonMailingStreet     = '')
  AND (acc.PersonMailingCity       IS NULL OR acc.PersonMailingCity       = '')
  AND (acc.PersonMailingPostalCode IS NULL OR acc.PersonMailingPostalCode = '')
  AND (acc.PersonMailingCountry    IS NULL OR acc.PersonMailingCountry    = '')
"""
POP_A = f"""{INVEST} AND {BLOCK_EMPTY}
  AND acc.BillingStreet IS NOT NULL AND acc.BillingStreet <> ''"""
POP_B = f"""{INVEST} AND {BLOCK_EMPTY}
  AND (acc.BillingStreet IS NULL OR acc.BillingStreet = '')
  AND acc.BillingCountryCode__c IS NOT NULL AND acc.BillingCountryCode__c <> ''"""
POP_C = INVEST + " AND acc.PersonMailingCountry REGEXP '^[A-Za-z]{2}$'"
POP_D = f"""{INVEST}
  AND (acc.PersonMailingCountry IS NULL OR acc.PersonMailingCountry = '')
  AND (   (acc.PersonMailingStreet     IS NOT NULL AND acc.PersonMailingStreet     <> '')
       OR (acc.PersonMailingCity       IS NOT NULL AND acc.PersonMailingCity       <> '')
       OR (acc.PersonMailingPostalCode IS NOT NULL AND acc.PersonMailingPostalCode <> ''))"""

# Suspect-country expression (05 query 3): the billing code contradicts the
# postal-code shape - eyeballing showed these are genuinely WRONG codes
# ('AT' with Wolfsburg/Berlin, 'DE' with Wien, 'AT' with Zadar). Their country
# is suppressed on copy; street/city/postal still move as one block.
SUSPECT_COUNTRY = """(
       (acc.BillingCountryCode__c IN ('AT','CH') AND acc.BillingPostalCode <> ''
        AND acc.BillingPostalCode NOT REGEXP '^[0-9]{4}$')
    OR (acc.BillingCountryCode__c IN ('DE','IT') AND acc.BillingPostalCode <> ''
        AND acc.BillingPostalCode NOT REGEXP '^[0-9]{5}$')
)"""

# Contract from the 05 recon run (mirror 2026-08-26 13:51).
# Update ONLY from a re-run recon.
EXPECTED = {"A": 4467, "B": 440, "C": 1230, "D": 8, "A_suspect": 108}

counts = {}
for pop, where in [("A", POP_A), ("B", POP_B), ("C", POP_C), ("D", POP_D),
                   ("A_suspect", f"{POP_A} AND {SUSPECT_COUNTRY}")]:
    counts[pop] = db.fetch_one(f"SELECT COUNT(*) AS n FROM crm_person_account_sfid_prod acc WHERE {where}")["n"]
    print(f"pop {pop}: {counts[pop]:,}   (expected {EXPECTED[pop]})")

for pop in counts:
    assert EXPECTED[pop] is not None, "fill EXPECTED from the recon before staging"
    assert counts[pop] == EXPECTED[pop], f"pop {pop}: {counts[pop]} != expected {EXPECTED[pop]}"
print("population counts match the recon contract")

## 4. Bookkeeping columns + batch guard

`_mailing_processed_at` (loader writeback, resumable) and `_mailing_prev_country`
(the original ISO-2 code for population C — the loader's live check refuses to
overwrite if the live value no longer equals it). Refuses to run if the batch id
already has rows.

In [ ]:
for col_name, ddl in [
    ("_mailing_processed_at", "ADD COLUMN _mailing_processed_at DATETIME NULL"),
    ("_mailing_prev_country", "ADD COLUMN _mailing_prev_country VARCHAR(8) NULL"),
]:
    n = db.fetch_one("""
        SELECT COUNT(*) AS n FROM information_schema.columns
        WHERE table_schema = DATABASE()
          AND table_name = 'crm_imp_person_accounts'
          AND column_name = %s
    """, (col_name,))["n"]
    if not n:
        db.execute(f"ALTER TABLE crm_imp_person_accounts {ddl}")
        print(f"column {col_name} added")
    else:
        print(f"column {col_name} exists")

existing = db.fetch_one(
    "SELECT COUNT(*) AS n FROM crm_imp_person_accounts WHERE _batch_id = %s",
    (BATCH_ID,),
)["n"]
assert existing == 0, f"{existing} rows already staged under this batch id — not re-inserting"
print("batch id is free")

## 5. Population A — full billing block copy

All four mailing fields from the same billing record: street, city, postal code,
and the country **name** mapped from `BillingCountryCode__c`. Two deliberate
country suppressions (address still copies; the rows just make no country claim):

- billing code **empty** (~11 rows) — nothing to map;
- billing code **suspect** (~108 rows, `SUSPECT_COUNTRY`) — the code contradicts
  the postal shape, i.e. it is wrong in the source; exported to
  `local_data/invest_mailing_suspect_country_review.csv` for manual follow-up.

In [ ]:
inserted_a = db.execute(f"""
    INSERT INTO crm_imp_person_accounts
        (_operation, _batch_id, _excluded, source, last_name, email,
         sf_account_id, sf_person_contact_id, external_id,
         address, city, postal_code, country)
    SELECT
        'update', %s, 0, 'invest_mailing_A', acc.LastName, acc.PersonEmail,
        acc.Id, acc.PersonContactId, acc.ExternalID__pc,
        acc.BillingStreet, NULLIF(acc.BillingCity, ''), NULLIF(acc.BillingPostalCode, ''),
        CASE WHEN {SUSPECT_COUNTRY} THEN NULL ELSE cn.name END
    FROM crm_person_account_sfid_prod acc
    LEFT JOIN crm_tmp_country_names cn ON cn.code = UPPER(acc.BillingCountryCode__c)
    WHERE {POP_A}
""", (BATCH_ID,))
print(f"staged A: {inserted_a:,} (expected {EXPECTED['A']:,})")
assert inserted_a == EXPECTED["A"]

no_country = db.fetch_one("""
    SELECT COUNT(*) AS n FROM crm_imp_person_accounts
    WHERE _batch_id = %s AND source = 'invest_mailing_A' AND (country IS NULL OR country = '')
""", (BATCH_ID,))["n"]
print(f"A rows staged without a country (empty or suspect billing code): {no_country:,}")

suspects = db.fetch_df(f"""
    SELECT acc.Id, acc.PersonEmail, acc.BillingStreet, acc.BillingCity,
           acc.BillingPostalCode, acc.BillingCountryCode__c
    FROM crm_person_account_sfid_prod acc
    WHERE {POP_A} AND {SUSPECT_COUNTRY}
""")
out = Path.cwd().parent / "local_data" / "invest_mailing_suspect_country_review.csv"
out.parent.mkdir(parents=True, exist_ok=True)
suspects.to_csv(out, index=False)
print(f"{len(suspects):,} suspect-country rows exported to {out} (expected {EXPECTED['A_suspect']})")
assert len(suspects) == EXPECTED["A_suspect"]

## 6. Population B — country name only

Mailing block fully empty, no billing street to copy, but a billing country code
exists. Only `PersonMailingCountry` gets the mapped name — there is no city that
could contradict it.

In [ ]:
inserted_b = db.execute(f"""
    INSERT INTO crm_imp_person_accounts
        (_operation, _batch_id, _excluded, source, last_name, email,
         sf_account_id, sf_person_contact_id, external_id, country)
    SELECT
        'update', %s, 0, 'invest_mailing_B', acc.LastName, acc.PersonEmail,
        acc.Id, acc.PersonContactId, acc.ExternalID__pc, cn.name
    FROM crm_person_account_sfid_prod acc
    JOIN crm_tmp_country_names cn ON cn.code = UPPER(acc.BillingCountryCode__c)
    WHERE {POP_B}
""", (BATCH_ID,))
print(f"staged B: {inserted_b:,} (expected {EXPECTED['B']:,})")
assert inserted_b == EXPECTED["B"]

## 7. Population C — in-place code→name conversion

`PersonMailingCountry` currently holds a bare ISO-2 code; it becomes the name.
The address is untouched. The original code is kept in `_mailing_prev_country`:
the loader refuses to write if the live value has changed since this snapshot.
Includes the billing≠mailing conflict accounts — **mailing wins**, only its
format changes. Values already spelled as names (e.g. "Austria") are not in
this population — nothing to convert.

In [ ]:
inserted_c = db.execute(f"""
    INSERT INTO crm_imp_person_accounts
        (_operation, _batch_id, _excluded, source, last_name, email,
         sf_account_id, sf_person_contact_id, external_id,
         country, _mailing_prev_country)
    SELECT
        'update', %s, 0, 'invest_mailing_C', acc.LastName, acc.PersonEmail,
        acc.Id, acc.PersonContactId, acc.ExternalID__pc,
        cn.name, acc.PersonMailingCountry
    FROM crm_person_account_sfid_prod acc
    JOIN crm_tmp_country_names cn ON cn.code = UPPER(acc.PersonMailingCountry)
    WHERE {POP_C}
""", (BATCH_ID,))
print(f"staged C: {inserted_c:,} (expected {EXPECTED['C']:,})")
assert inserted_c == EXPECTED["C"]

conv = db.fetch_all("""
    SELECT _mailing_prev_country AS code, country AS name, COUNT(*) AS n
    FROM crm_imp_person_accounts
    WHERE _batch_id = %s AND source = 'invest_mailing_C'
    GROUP BY 1, 2 ORDER BY n DESC
""", (BATCH_ID,))
print("\nconversions:")
for r in conv:
    print(f"  {r['code']} -> {r['name']}: {int(r['n']):,}")

## 8. Population D — review export (no writes)

Partial mailing blocks: street/city from some import, country empty. Filling the
country from billing here is exactly how a Vienna/Germany mismatch would be born,
so these are **never staged**. PII, lands in the gitignored `local_data/`.

In [ ]:
review = db.fetch_df(f"""
    SELECT acc.Id, acc.PersonEmail, acc.PersonMailingStreet, acc.PersonMailingCity,
           acc.PersonMailingPostalCode, acc.BillingCountryCode__c, acc.BillingCity,
           acc.SourceSystem__pc, acc.LastModifiedDate
    FROM crm_person_account_sfid_prod acc
    WHERE {POP_D}
""")
out = Path.cwd().parent / "local_data" / "invest_mailing_partial_review.csv"
out.parent.mkdir(parents=True, exist_ok=True)
review.to_csv(out, index=False)
print(f"{len(review):,} partial-block accounts exported to {out} (expected {EXPECTED['D']})")
assert len(review) == EXPECTED["D"]

## 9. Final contract

One row per account across the whole batch, populations disjoint, every country
value a full name (never a bare code), C rows carry their original code. These
numbers are the load contract for `07_run_mailing_country.ipynb`.

In [ ]:
summary = db.fetch_one("""
    SELECT COUNT(*) AS rows_staged,
           COUNT(DISTINCT sf_account_id) AS accounts,
           SUM(country REGEXP '^[A-Za-z]{2}$') AS bare_codes,
           SUM(source = 'invest_mailing_A') AS a_rows,
           SUM(source = 'invest_mailing_B') AS b_rows,
           SUM(source = 'invest_mailing_C') AS c_rows,
           SUM(source = 'invest_mailing_C' AND (_mailing_prev_country IS NULL OR _mailing_prev_country = '')) AS c_without_prev,
           SUM(source IN ('invest_mailing_B','invest_mailing_C') AND (country IS NULL OR country = '')) AS bc_without_country
    FROM crm_imp_person_accounts
    WHERE _batch_id = %s
""", (BATCH_ID,))
for k, v in summary.items():
    print(f"{k:20s} {int(v):,}")

assert int(summary["rows_staged"]) == int(summary["accounts"]), "duplicate sf_account_id staged"
assert int(summary["rows_staged"]) == EXPECTED["A"] + EXPECTED["B"] + EXPECTED["C"]
assert int(summary["bare_codes"]) == 0, "bare ISO-2 code staged as country"
assert int(summary["c_without_prev"]) == 0, "C row without its original code"
assert int(summary["bc_without_country"]) == 0, "B/C row without a country name"

dist = db.fetch_all("""
    SELECT country, COUNT(*) AS n FROM crm_imp_person_accounts
    WHERE _batch_id = %s GROUP BY 1 ORDER BY n DESC
""", (BATCH_ID,))
print("\ncountry distribution:")
for r in dist:
    print(f"  {r['country'] or '(none, address-only A rows)'}: {int(r['n']):,}")

# The mapping table was only needed for the INSERTs above - the staged rows
# carry the names themselves, the loader and the archive never read it.
# (Re-running the notebook recreates it in section 2.)
db.execute("DROP TABLE IF EXISTS crm_tmp_country_names")
print("\ncrm_tmp_country_names dropped - staging frozen")

## Next

- Dry-run: `python nationality-backfill/update_invest_mailing.py <batch_id> --dry-run`
  (or via `07_run_mailing_country.ipynb`, which drives all later phases).
- Probes: one A account and one C account (07, `RUN_PROBE` gate — C overwrites an
  existing value, so its before/after gets eyeballed first).
- Bulk load: **does not run without Arsal's explicit go-ahead** (07, `RUN_LOAD` gate).